Yahoo Finance
      │
      ▼
Pandas DataFrame
      │
      ├── Metadata
      │      Company
      │      Statement Type
      │      Frequency
      │      Fiscal Year
      │
      ├── Convert each row into a document
      │
      ▼
Embedding Model
      │
      ▼
Vector Database
      │
      ▼
LLM / Finance Assistant

In [1]:
!pip install yfinance
!pip install chromadb
!pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 718.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [2]:
import os
import requests
import pandas as pd
import yfinance as yf
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
from datetime import datetime, timedelta

In [3]:
import chromadb

client = chromadb.PersistentClient(path="FinanceKnowledgeDB")

collection = client.get_or_create_collection(
    name="financial_statements"
)

In [4]:
import time

exchange_type = ['NSE', 'BSE']
#Tickers for Indian IT Companies in Yahoo Finance
#If the ticker has .NS as suffix then the data is taken from NSE
ind_it_cos = [
    "INFY.NS",
    "TCS.NS",
    "HCLTECH.NS",
    "WIPRO.NS",
    "LTM.NS",
]
company_names = [company.replace('.NS', '').replace('.BO', '') for company in ind_it_cos]
kind_of_stmt = ['Income_Statement', 'Balance_Sheet', 'Cash_Flow']
stmt_freq = ['Quarterly', 'Annual']

# Button that bulk-loads every company / statement / frequency combination
load_all_button = widgets.Button(
    description='Load All Companies',
    disabled=False,
    button_style='success',
    tooltip='Fetch and store financial statements for every company in the list',
    icon='cloud-download',
)

progress_bar = widgets.IntProgress(
    value=0, min=0, max=1, description='Idle:', bar_style='',
    layout=widgets.Layout(width='50%'),
)
progress_label = widgets.Label(value='')
load_status = widgets.Output()
result_output = widgets.Output()

# Cache Ticker objects so we don't re-hit Yahoo Finance for `.info` on every
# (statement, frequency) combination for the same company.
_yf_ticker_cache = {}


def get_ticker(ticker_symbol):
    if ticker_symbol not in _yf_ticker_cache:
        _yf_ticker_cache[ticker_symbol] = yf.Ticker(ticker_symbol)
    return _yf_ticker_cache[ticker_symbol]


def store_financial_statement(df, company_ticker, company_full_name, statement, frequency, collection):
    """Convert one financial-statement DataFrame into per-metric documents
    and upsert them into the Chroma collection.

    Stores BOTH `ticker` (short name, e.g. "INFY" — matches the dropdown
    values so results can be looked up reliably) and `company` (the full
    display name, e.g. "Infosys Limited") in the metadata, since the two are
    not interchangeable.
    """
    if df is None or df.empty:
        return 0

    documents, metadatas, ids = [], [], []
    latest_period = df.columns[-1]
    year = latest_period.year if hasattr(latest_period, "year") else str(latest_period)

    for metric in df.index:
        value = df.loc[metric, latest_period]
        if pd.isna(value):
            continue  # skip metrics with no reported value for this period

        value = float(value)
        document = (
            f"Company : {company_full_name}\n"
            f"Statement : {statement}\n"
            f"Frequency : {frequency}\n"
            f"Year : {year}\n"
            f"Metric : {metric}\n"
            f"Value : {value}"
        )
        documents.append(document)
        metadatas.append({
            "ticker": company_ticker,       # used to filter/match dropdown selection
            "company": company_full_name,   # used for display
            "statement": statement,
            "frequency": frequency,
            "year": year,
            "metric": metric,
            "value": value,
        })
        ids.append(f"{company_ticker}_{statement}_{frequency}_{year}_{metric}")

    if not documents:
        return 0

    # upsert (not add) so re-running the bulk load overwrites old values
    # instead of raising a duplicate-id error.
    collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
    return len(documents)


def load_and_save_financial_data(selected_company_name, selected_kind_of_stmt, selected_freq):
    ticker_symbol = f"{selected_company_name}.NS"
    company_yfinance = get_ticker(ticker_symbol)

    try:
        company_full_name = company_yfinance.info.get('longName', selected_company_name)
    except Exception:
        company_full_name = selected_company_name

    # Only fetch the single statement actually requested — not all six.
    statement_fetchers = {
        ("Annual", "Income_Statement"): lambda: company_yfinance.income_stmt,
        ("Annual", "Balance_Sheet"): lambda: company_yfinance.balance_sheet,
        ("Annual", "Cash_Flow"): lambda: company_yfinance.cashflow,
        ("Quarterly", "Income_Statement"): lambda: company_yfinance.quarterly_income_stmt,
        ("Quarterly", "Balance_Sheet"): lambda: company_yfinance.quarterly_balance_sheet,
        ("Quarterly", "Cash_Flow"): lambda: company_yfinance.quarterly_cashflow,
    }

    df = statement_fetchers[(selected_freq, selected_kind_of_stmt)]()
    return store_financial_statement(
        df, selected_company_name, company_full_name, selected_kind_of_stmt, selected_freq, collection
    )


def load_all_companies_data(companies=None, statements=None, freqs=None, delay=0.5):
    """Loop over every company / statement-type / frequency combination and
    store each into the Chroma collection, updating a progress bar as it goes.

    `delay` adds a short pause between Yahoo Finance calls to reduce the
    chance of being rate-limited when pulling ~100 requests back to back.
    """
    companies = companies or company_names
    statements = statements or kind_of_stmt
    freqs = freqs or stmt_freq

    combos = [(c, s, f) for c in companies for s in statements for f in freqs]
    total = len(combos)

    progress_bar.max = total
    progress_bar.value = 0
    progress_bar.bar_style = ''
    progress_bar.description = 'Loading:'

    stored_count = 0
    failed = []

    with load_status:
        load_status.clear_output()
        for i, (company, statement, freq) in enumerate(combos, start=1):
            progress_label.value = f"{i}/{total} — {company}: {statement} ({freq})"
            try:
                stored_count += load_and_save_financial_data(company, statement, freq)
            except Exception as e:
                failed.append((company, statement, freq, str(e)))
            progress_bar.value = i
            time.sleep(delay)  # be polite to Yahoo Finance's rate limits

        progress_bar.bar_style = 'success' if not failed else 'warning'
        progress_label.value = f"Done — {total - len(failed)}/{total} combinations loaded."
        print(f"Stored {stored_count} metric records across {len(companies)} companies.")
        if failed:
            print(f"\n{len(failed)} combination(s) failed to load:")
            for company, statement, freq, err in failed:
                print(f"  - {company} / {statement} / {freq}: {err}")


def display_selected_result(b=None):
    """Look up the data already stored for the current dropdown selection
    (company / statement / frequency) and render it as a table — no network
    call, since this reads straight from the Chroma collection."""
    sel_company = dropdown1.value
    sel_kind_of_stmt = dropdown2.value
    sel_stmt_freq = dropdown3.value

    with result_output:
        result_output.clear_output()
        results = collection.get(
            where={
                "$and": [
                    {"ticker": {"$eq": sel_company}},
                    {"statement": {"$eq": sel_kind_of_stmt}},
                    {"frequency": {"$eq": sel_stmt_freq}},
                ]
            }
        )

        metadatas = results.get("metadatas") or []
        if not metadatas:
            print(
                f"No stored data yet for {sel_company} / {sel_kind_of_stmt} / {sel_stmt_freq}.\n"
                "Click 'Load All Companies' first."
            )
            return

        year = metadatas[0].get("year")
        company_full_name = metadatas[0].get("company", sel_company)

        table = pd.DataFrame(
            [{"Metric": m["metric"], "Value": m["value"]} for m in metadatas]
        ).sort_values("Metric").reset_index(drop=True)

        print(f"{company_full_name} ({sel_company}) — {sel_kind_of_stmt}, {sel_stmt_freq}, Year {year}")
        display(table)


# Wire up buttons
load_all_button.on_click(lambda b: load_all_companies_data())
display(load_all_button, widgets.HBox([progress_bar, progress_label]), load_status)


Button(button_style='success', description='Load All Companies', icon='cloud-download', style=ButtonStyle(), t…

Output()

## Part 2 — Conversational RAG Finance Assistant (Retrieval + Prompt Engineering + Memory + LLM)

The section above builds the knowledge base: Yahoo Finance data is chunked
into one document per metric and stored in the `financial_statements` Chroma
collection with rich metadata (`ticker`, `statement`, `frequency`, `year`,
`metric`, `value`).

The cells below turn that knowledge base into a real **Retrieval-Augmented
Generation (RAG) assistant**:

```
User question
      │
      ▼
Conversation Memory  ──►  expand query with recent turns (for "it", "that company", etc.)
      │
      ▼
Metadata Filter Extraction  (detects company / statement type / frequency mentioned)
      │
      ▼
Chroma Vector Search  (semantic retrieval of the most relevant metric records)
      │
      ▼
Prompt Engineering  (system prompt + retrieved context + chat history + question)
      │
      ▼
LLM (Gemini via Google Generative AI API)
      │
      ▼
Answer  ──► appended back into Conversation Memory for the next turn
```

Run the cells in order, then use the chat box at the bottom to ask things like:

- *"What was Infosys's total revenue in the latest quarter?"*
- *"Compare TCS and Wipro net income."*
- *"And what about their balance sheet?"* (memory resolves "their" from the previous turn)


In [ ]:
!pip install google.generativeai -q


In [ ]:
import os
import getpass
import google.generativeai as genai
from google.api_core.exceptions import Unauthenticated, PermissionDenied

# ----------------------------------------------------------------------------
# LLM client setup
# ----------------------------------------------------------------------------
# Looks for the key in this order:
#   1. an already-set GEMINI_API_KEY (or GOOGLE_API_KEY) environment variable
#   2. a Colab secret named GEMINI_API_KEY (Colab -> key icon in sidebar)
#   3. an interactive, hidden prompt (getpass) as a last resort
#
# Get a key at https://aistudio.google.com/apikey — a Cloud OAuth client ID
# or a Vertex AI service-account key will NOT work here.
LLM_MODEL = "gemini-2.0-flash"  # swap for another Gemini model string if you prefer

_api_key_configured = False


def _resolve_api_key():
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if api_key:
        return api_key.strip()

    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key:
            return api_key.strip()
    except Exception:
        pass

    return getpass.getpass("Enter your Gemini API key: ").strip()


def reset_gemini_key():
    """Call this if you got a 401/permission error — clears the cached key
    so the next call_llm() re-prompts instead of silently reusing a bad one."""
    global _api_key_configured
    _api_key_configured = False
    os.environ.pop("GEMINI_API_KEY", None)
    os.environ.pop("GOOGLE_API_KEY", None)
    print("Cleared cached Gemini API key. The next question will prompt for a new one.")


def _ensure_configured():
    """Configure the google-generativeai SDK with an API key exactly once."""
    global _api_key_configured
    if not _api_key_configured:
        api_key = _resolve_api_key()
        os.environ["GEMINI_API_KEY"] = api_key
        genai.configure(api_key=api_key)
        _api_key_configured = True


def call_llm(system_prompt, user_prompt, max_tokens=1024):
    """Single point of contact with the LLM — keeps prompt engineering and
    error handling in one place."""
    _ensure_configured()
    try:
        model = genai.GenerativeModel(LLM_MODEL, system_instruction=system_prompt)
        response = model.generate_content(
            user_prompt,
            generation_config=genai.types.GenerationConfig(max_output_tokens=max_tokens),
        )
        return response.text.strip()
    except (Unauthenticated, PermissionDenied) as e:
        # Don't leave a bad key cached — next call will re-prompt.
        reset_gemini_key()
        return (
            "⚠️ Authentication with the Gemini API failed (401/403). This means the "
            "API key itself was rejected — not a bug in the retrieval pipeline. "
            "Common fixes:\n"
            "  1. Get a key from https://aistudio.google.com/apikey (Google Cloud "
            "OAuth credentials and Vertex AI service-account keys won't work here).\n"
            "  2. Make sure the key has no extra spaces/quotes when pasted.\n"
            "  3. If the key was created in Cloud Console with API restrictions, "
            "make sure 'Generative Language API' is allowed.\n"
            f"Original error: {e}\n"
            "I've cleared the cached key — just ask your question again to re-enter it."
        )
    except Exception as e:
        return f"⚠️ LLM call failed: {e}"


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# ----------------------------------------------------------------------------
# Conversation memory
# ----------------------------------------------------------------------------
# Keeps a rolling window of the last few turns so the assistant can resolve
# follow-up questions ("What about last quarter?", "and Wipro?") and so the
# LLM can see what it already told the user.

class ConversationMemory:
    def __init__(self, max_turns=6):
        self.max_turns = max_turns          # number of USER turns retained
        self.turns = []                     # [{"role": "user"/"assistant", "content": str}, ...]

    def add(self, role, content):
        self.turns.append({"role": role, "content": content})
        # Trim to the most recent `max_turns` user+assistant pairs
        max_messages = self.max_turns * 2
        if len(self.turns) > max_messages:
            self.turns = self.turns[-max_messages:]

    def get_recent_user_queries(self, n=2):
        user_msgs = [t["content"] for t in self.turns if t["role"] == "user"]
        return user_msgs[-n:]

    def format_history(self):
        if not self.turns:
            return "(no previous turns yet)"
        lines = []
        for t in self.turns:
            speaker = "User" if t["role"] == "user" else "Assistant"
            lines.append(f"{speaker}: {t['content']}")
        return "\n".join(lines)

    def clear(self):
        self.turns = []


memory = ConversationMemory(max_turns=6)


In [ ]:
# ----------------------------------------------------------------------------
# Retrieval pipeline
# ----------------------------------------------------------------------------
# 1) build_retrieval_query — folds a bit of recent conversation into the
#    search text, so pronoun-style follow-ups still retrieve well.
# 2) extract_filters — looks for an explicit company / statement-type /
#    frequency mention in the question and turns it into a Chroma `where`
#    filter, so retrieval isn't purely semantic when the user is specific.
# 3) retrieve_context — runs the actual vector search against the
#    `financial_statements` collection built in Part 1.

def build_retrieval_query(current_query, memory, history_terms=2):
    recent = memory.get_recent_user_queries(history_terms)
    return " ".join(recent + [current_query])


def extract_filters(query):
    q = query.lower()
    filters = []

    matched_companies = [c for c in company_names if c.lower() in q]
    if len(matched_companies) == 1:
        filters.append({"ticker": {"$eq": matched_companies[0]}})

    stmt_keywords = {
        "Income_Statement": ["income statement", "revenue", "profit", "net income", "earnings", "margin"],
        "Balance_Sheet": ["balance sheet", "assets", "liabilities", "equity"],
        "Cash_Flow": ["cash flow", "cashflow", "operating cash", "free cash flow"],
    }
    matched_stmts = [s for s, kws in stmt_keywords.items() if any(kw in q for kw in kws)]
    if len(matched_stmts) == 1:
        filters.append({"statement": {"$eq": matched_stmts[0]}})

    if "quarter" in q:
        filters.append({"frequency": {"$eq": "Quarterly"}})
    elif any(k in q for k in ["annual", "yearly", "fiscal year", "full year"]):
        filters.append({"frequency": {"$eq": "Annual"}})

    if not filters:
        return None
    if len(filters) == 1:
        return filters[0]
    return {"$and": filters}


def retrieve_context(query, memory, n_results=10):
    retrieval_query = build_retrieval_query(query, memory)
    where_filter = extract_filters(query)

    try:
        results = collection.query(
            query_texts=[retrieval_query],
            n_results=n_results,
            where=where_filter,
        )
    except Exception:
        # If the filter combination matches nothing (or errors), fall back
        # to an unfiltered semantic search rather than failing outright.
        results = collection.query(query_texts=[retrieval_query], n_results=n_results)

    docs = (results.get("documents") or [[]])[0]
    metas = (results.get("metadatas") or [[]])[0]
    return docs, metas


def format_context_block(docs):
    if not docs:
        return "No matching records were found in the vector database."
    return "\n---\n".join(docs)


In [ ]:
# ----------------------------------------------------------------------------
# Prompt engineering
# ----------------------------------------------------------------------------
# The system prompt fixes the assistant's role and hard rules (grounding,
# honesty about missing data, formatting). The user-turn prompt is assembled
# fresh each call from: conversation history + retrieved context + question —
# this is the "augmentation" step of RAG.

SYSTEM_PROMPT = """You are a Finance Knowledge Assistant specialised in the financial \
statements of Indian IT services companies (Infosys, TCS, HCL Tech, Wipro, and similar \
firms tracked in this notebook's Chroma vector database).

Follow these rules on every turn:
1. Ground every figure you state in the "Retrieved Context" block provided below. \
Never invent, estimate, or infer a financial number that isn't present in the context.
2. If the retrieved context doesn't contain what's needed to answer, say so plainly and \
suggest the user click "Load All Companies" first, or rephrase the question with a \
specific company/statement/period.
3. When comparing companies or periods, present the numbers in a small markdown table \
rather than a wall of prose.
4. Always state which company, statement type, frequency, and year each figure comes from.
5. Use the conversation history to resolve references like "it", "that company", \
"the same quarter", or "what about last year".
6. Be concise by default; expand only when the user asks for more detail or explanation.
7. You are not a licensed financial advisor. For requests for investment advice or buy/sell \
recommendations, share the relevant factual context only and note that you can't give \
personalised financial advice.
"""


def build_user_prompt(query, context_block, history_text):
    return f"""Conversation so far:
{history_text}

Retrieved Context (from the financial_statements vector database):
{context_block}

Current Question: {query}

Answer the current question, following the system rules above.
"""


In [ ]:
# ----------------------------------------------------------------------------
# Full RAG pipeline: retrieval -> prompt engineering -> LLM -> memory update
# ----------------------------------------------------------------------------

def rag_answer(query, n_results=10):
    docs, metas = retrieve_context(query, memory, n_results=n_results)
    context_block = format_context_block(docs)
    history_text = memory.format_history()

    user_prompt = build_user_prompt(query, context_block, history_text)
    answer = call_llm(SYSTEM_PROMPT, user_prompt)

    memory.add("user", query)
    memory.add("assistant", answer)
    return answer, docs, metas


In [ ]:
# ----------------------------------------------------------------------------
# Chat UI
# ----------------------------------------------------------------------------
query_box = widgets.Text(
    placeholder="Ask about revenue, margins, balance sheet items, comparisons...",
    description="Ask:",
    layout=widgets.Layout(width="80%"),
)
ask_button = widgets.Button(description="Ask", button_style="primary", icon="comment")
clear_button = widgets.Button(description="Clear Chat", button_style="warning", icon="trash")
chat_output = widgets.Output()


def render_conversation():
    with chat_output:
        chat_output.clear_output()
        if not memory.turns:
            print("Ask a question about any of the loaded companies' financial statements to get started.")
            return
        for t in memory.turns:
            speaker = "🧑 You" if t["role"] == "user" else "🤖 Assistant"
            print(f"{speaker}: {t['content']}\n")


def on_ask_clicked(b=None):
    query = query_box.value.strip()
    if not query:
        return
    query_box.value = ""
    rag_answer(query)
    render_conversation()


def on_clear_clicked(b=None):
    memory.clear()
    render_conversation()


ask_button.on_click(on_ask_clicked)
clear_button.on_click(on_clear_clicked)
query_box.on_submit(on_ask_clicked)  # Enter key also asks

display(widgets.HTML("<h3>💬 Ask the Finance Assistant</h3>"))
display(widgets.HBox([query_box, ask_button, clear_button]))
display(chat_output)
render_conversation()


HTML(value='<h3>💬 Ask the Finance Assistant</h3>')

Output()